In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
seeds = [42, 67, 99, 70, 73]

ptbxl_dirs = {
    17418: "runs-ptbxl",
    8722: "runs-ptbxl-8k",
    4356: "runs-ptbxl-4k",
    2175: "runs-ptbxl-2k",
    1091: "runs-ptbxl-1k",
    547: "runs-ptbxl-512",
    273: "runs-ptbxl-256",
}

experiments = {
    "ProtoSSL HEEDB": "protossl-heedb-pila",
    "ProtoSSL HEEDB (PIA)": "protossl-heedb-pia",
}

In [ ]:
data = []
base_path = Path("/opt/gpu_working/steven")
for seed in seeds:
    seed_path = base_path / f"protossl-outputs-seed{seed}"
    for subset_size, ptbxl_dir in ptbxl_dirs.items():
        run_dir = seed_path / ptbxl_dir
        for exp_name, exp_dir in experiments.items():
            model_dir = run_dir / exp_dir
            summary_json = model_dir / "learn-prototype-assignments/latest/wandb/latest-run/files/wandb-summary.json"
            if not summary_json.exists(): # TODO remove this
                continue
            with open(summary_json, "r") as f:
                summary = json.load(f)
            runtime_sec = summary["_runtime"]
            datum = {
                "Seed": seed,
                "Train Size": subset_size,
                "Model": exp_name,
                "Runtime": runtime_sec,
            }
            data.append(datum)
data = pd.DataFrame(data)

In [ ]:
data.pivot(columns=["Train Size"], index=["Model", "Seed"], values=["Runtime"])

In [ ]:
pd.DataFrame(data)[["Train Size", "Model", "Runtime"]].groupby(["Train Size", "Model"]).mean()